In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Gradient Penalties and Convergence

The garment generator of that section learned to draw shirts, shoes and dresses, but its losses never settled. The critic's loss stayed well below $\log 2$ and the generator's well above it, both still drifting when training stopped. At the equilibrium, where the generated images match the data and the critic can only guess, both losses equal $\log 2$. The last three sections computed what the game is worth when the critic answers optimally. Training never gives the critic that chance: both networks take small gradient steps at the same time, and each step of one player changes the objective of the other. Knowing where the equilibrium lies does not tell us whether these steps lead there.

@Mescheder.Geiger.Nowozin.2018 asked which training methods for GANs converge. Their answer starts from the smallest game there is: one data point, one generated point and a straight-line critic. Plain gradient steps fail even on this game, and a penalty on the critic's slope at the data and generated samples repairs them. We analyze that game, add the penalty, ask what the penalized game measures, and train the resulting R3GAN loss [@Huang.Gokaslan.Kuleshov.ea.2024] on a grid of 25 Gaussians.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The Dirac-GAN

The smallest game can be solved exactly, and it shows both obstacles of this section: its value is flat in the generator's position, and gradient steps rotate around the solution instead of approaching it.

### The Game on Two Points

Place the data at the origin and let the generator produce a single point on the real line: $p = \delta_0$ and $q_\theta = \delta_\theta$, so $q = p$ exactly when $\theta = 0$. Let the critic be linear, $D_\psi(x) = \psi x$. that section scored a real sample $x$ by $\ell(D(x))$ and a generated sample $x'$ by $\ell(-D(x'))$ for a classification loss $\ell$, giving the margin objective

$$
V_\ell(D) = \mathbb{E}_{x \sim p}\big[\ell(D(x))\big] + \mathbb{E}_{x' \sim q}\big[\ell(-D(x'))\big].
$$

For two point masses both expectations are single evaluations:

$$
V(\theta, \psi) \;=\; \ell\big(D_\psi(0)\big) + \ell\big({-D_\psi(\theta)}\big)
\;=\; \ell(0) + \ell(-\psi\theta).
$$

This is the *Dirac-GAN* of @Mescheder.Geiger.Nowozin.2018 . Any differentiable loss with $\ell'(0) > 0$ works. We use the logistic loss $\ell = \log\sigma$, whose derivatives are $\ell'(t) = \sigma(-t)$ and $\ell''(t) = -\sigma(t)\sigma(-t)$, so $\ell'(0) = \tfrac12$ and $\ell''(0) = -\tfrac14$.

For every $\theta \neq 0$ the two point masses have disjoint supports. The critic's best response sends $\psi\theta \to -\infty$, so $\sup_\psi V = \ell(0) + \sup_t \ell(t)$, the same number for every $\theta \neq 0$. A value that does not depend on $\theta$ gives the generator no gradient, however close to the data it starts. This is the saturation of that section in its smallest form.

### Circles in Continuous Time

Training does not compute a best response. Both players take gradient steps from where they are. The game has one stationary point, $(\theta, \psi) = (0, 0)$, where the generated point matches the data and the critic is flat. In the simultaneous gradient flow the generator descends $V$ and the critic ascends it:

$$
\dot\theta \;=\; -\partial_\theta V \;=\; \psi\, \ell'(-\psi\theta),
\qquad
\dot\psi \;=\; +\partial_\psi V \;=\; -\theta\, \ell'(-\psi\theta).
$$

Both components are the same positive scalar $\ell'(-\psi\theta)$ times the vector $(\psi, -\theta)$, which is orthogonal to the position $(\theta, \psi)$. The squared distance to the equilibrium therefore does not change:

$$
\frac{d}{dt}\big(\theta^2 + \psi^2\big)
= 2\theta\psi\,\ell'(-\psi\theta) - 2\psi\theta\,\ell'(-\psi\theta) = 0 .
$$

Every trajectory is a circle around the solution, for every loss in this family. The linearization at the origin gives the same picture locally,

$$
J = \begin{pmatrix} 0 & \ell'(0) \\ -\ell'(0) & 0 \end{pmatrix},
\qquad
\nu_{1,2} = \pm\, i\, \ell'(0),
$$

with purely imaginary eigenvalues: the equilibrium is a center, not an attractor. The generated point passes the data again and again without settling there. The common factor $\ell'(-\psi\theta) = \sigma(D_\psi(\theta))$ is the weight with which the saturating generator of that section scales the update of a generated sample. Because it multiplies both components, it sets how fast the parameters turn and not where they go, so no reweighting of this kind turns the circle into a spiral toward the solution.

### Outward Spirals in Discrete Time

Discrete steps do worse than the flow. Write $\mathbf{u}(\theta, \psi)$ for the right-hand side of the equation. A simultaneous step of size $h > 0$ maps $(\theta, \psi)$ to $(\theta, \psi) + h\,\mathbf{u}$. Since $\mathbf{u}$ is orthogonal to $(\theta, \psi)$, the Pythagorean theorem gives

$$
\big\| (\theta, \psi) + h\, \mathbf{u} \big\|^2
\;=\; \big\| (\theta, \psi) \big\|^2 + h^2 \big\| \mathbf{u} \big\|^2 ,
$$

so each step adds $h^2\|\mathbf{u}\|^2$ to the squared distance. For the logistic loss $\mathbf{u}$ vanishes only at the equilibrium, so every step from any other point moves away from the solution, and a smaller step only slows the outward spiral (exercise 1). Updating the two players in turn instead of simultaneously does not restore convergence either (exercise 5).

The two repairs of the earlier sections leave these dynamics in place. The non-saturating generator has the same Jacobian $J$ at the equilibrium; away from it the field gains a slow inward drift, which discrete steps stop on a cycle of radius about $\sqrt{2h}$ (exercise 9). The pairing objective of that section scores the pair by $\ell\big(D_\psi(0) - D_\psi(\theta)\big) = \ell(-\psi\theta)$, which differs from the equation only by the constant $\ell(0)$, so its gradient field, its circles and its spirals are the classical ones. A cure has to change the dynamics themselves.

## Zero-Centered Penalties

The equilibrium attracts the iterates only if the eigenvalues of the linearized dynamics have negative real parts. The rotation needs friction, and the natural place to add it is the critic's slope, the quantity $\psi$ that keeps the rotation going. Define the two *zero-centered gradient penalties*

$$
R_1 \;=\; \frac{\gamma}{2}\, \mathbb{E}_{x \sim p}\Big[ \big\| \nabla_x D(x) \big\|^2 \Big],
\qquad
R_2 \;=\; \frac{\gamma}{2}\, \mathbb{E}_{x' \sim q}\Big[ \big\| \nabla_{x'} D(x') \big\|^2 \Big],
$$

with weight $\gamma > 0$, and let the critic maximize $V_\ell(D) - R_1 - R_2$ while the generator's objective stays unchanged [@Roth.Lucchi.Nowozin.ea.2017; @Mescheder.Geiger.Nowozin.2018]. The penalties act on the critic's gradient with respect to its *input*, not its parameters: $R_1$ limits how steeply the critic varies at data samples and $R_2$ at generated samples. They are called zero-centered because the flat critic $\nabla_x D = 0$ minimizes both. At $q = p$ the optimal critic $\log(p/q)$ of that section is the constant zero, so the penalties vanish at the equilibrium they are meant to stabilize. Since $\tfrac12(\mathbb{E}_p + \mathbb{E}_q) = \mathbb{E}_m$, their sum is

$$
R_1 + R_2 \;=\; \gamma\, \mathbb{E}_{x \sim m}\Big[ \big\| \nabla_x D(x) \big\|^2 \Big],
\qquad m = \frac{p + q}{2},
$$

one smoothness penalty under the mixture $m$, which constrains the critic wherever either distribution puts mass.

### Damping the Dirac-GAN

For the linear critic, $\nabla_x D_\psi = \psi$ at every $x$, so both penalties equal $\tfrac{\gamma}{2}\psi^2$. Subtracting one of them from the critic's objective adds $-\gamma\psi$ to the critic's update:

$$
\dot\theta = \psi\,\ell'(-\psi\theta),
\qquad
\dot\psi = -\theta\,\ell'(-\psi\theta) - \gamma\psi .
$$

Using both penalties replaces $\gamma$ by $2\gamma$. Only the lower-right entry of the Jacobian changes:

$$
J_\gamma = \begin{pmatrix} 0 & \ell'(0) \\ -\ell'(0) & -\gamma \end{pmatrix},
\qquad
\nu_{1,2} \;=\; -\frac{\gamma}{2} \pm \sqrt{ \frac{\gamma^2}{4} - \ell'(0)^2 } .
$$

Both eigenvalues have negative real parts for every $\gamma > 0$. The equilibrium attracts, and small enough gradient steps converge to it at a linear rate [@Mescheder.Geiger.Nowozin.2018]. As for a damped spring, there are three regimes. For $\gamma < 2\ell'(0)$ the square root is imaginary and trajectories spiral inward. At $\gamma = 2\ell'(0)$, which is $\gamma = 1$ for the logistic loss, the repeated real eigenvalue gives critical damping, the fastest approach without oscillation. Larger values overdamp the dynamics: the slower eigenvalue, close to $-\ell'(0)^2/\gamma$, makes convergence slower as $\gamma$ grows.

### Phase Portraits

We check the circles, the outward spiral and the damping by simulating the two equations for the logistic loss. The function `trajectory` repeats one update of step size $h$: `rk4_step` integrates the flow with a fourth-order Runge--Kutta step, and `euler_step` takes one simultaneous gradient step.

In [ ]:
def dirac_field(theta, psi, gamma=0.0):
    """Simultaneous gradient field of the Dirac-GAN with the logistic loss."""
    w = 1.0 / (1.0 + np.exp(-psi * theta))      # l'(-psi theta) = w_sat
    return np.array([psi * w, -theta * w - gamma * psi])

def euler_step(x, h, gamma):                     # one simultaneous step
    return x + h * dirac_field(*x, gamma)

def rk4_step(x, h, gamma):                       # one RK4 step of the ODE
    k1 = dirac_field(*x, gamma)
    k2 = dirac_field(*(x + h / 2 * k1), gamma)
    k3 = dirac_field(*(x + h / 2 * k2), gamma)
    k4 = dirac_field(*(x + h * k3), gamma)
    return x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def trajectory(step, x0, h, steps, gamma=0.0):
    xs = [np.array(x0)]
    for _ in range(steps):
        xs.append(step(xs[-1], h, gamma))
    return np.stack(xs)

All panels start at $(\theta, \psi) = (1, 1)$, in red; the cross marks the equilibrium. They show the flow, 300 simultaneous steps of size $h = 0.2$ against the flow's circle (dashed), and the same steps with the penalty at $\gamma = 0.3$, in the underdamped regime, and at the critical $\gamma = 1$.

In [ ]:
def phase_panel(ax, paths, title, lim, legend=None, fmts=('C0-', 'C1-')):
    """Draw paths in the (theta, psi) plane, with start and equilibrium."""
    X = [P[:, 0] for P in paths] + [[paths[0][0, 0]], [0.0]]
    Y = [P[:, 1] for P in paths] + [[paths[0][0, 1]], [0.0]]
    d2l.plot(X, Y, r'$\theta$', r'$\psi$', legend, xlim=(-lim, lim),
             ylim=(-lim, lim), fmts=fmts[:len(paths)] + ('C3o', 'k+'), axes=ax)
    ax.set_title(title)
    ax.set_aspect('equal')

x0, t = (1.0, 1.0), np.linspace(0, 2 * np.pi, 200)
circle = np.sqrt(2) * np.stack([np.cos(t), np.sin(t)], axis=1)
fig, axes = d2l.plt.subplots(1, 3, figsize=(9.5, 3.4))
phase_panel(axes[0], [trajectory(rk4_step, x0, 0.05, 800)], 'flow', 1.6)
phase_panel(axes[1], [trajectory(euler_step, x0, 0.2, 300), circle], 'steps',
            3.2, fmts=('C0-', 'k--'))
phase_panel(axes[2], [trajectory(euler_step, x0, 0.2, 300, gamma=0.3),
                      trajectory(euler_step, x0, 0.2, 300, gamma=1.0)],
            'penalized', 1.6, legend=[r'$\gamma = 0.3$', r'$\gamma = 1$'])
fig.tight_layout()

The integrated flow retraces one circle, as the conservation law requires. The discrete iterates leave the dashed circle at once and move outward on every revolution; after 300 steps their distance from the equilibrium has doubled. Both penalized trajectories converge, the underdamped one along a shrinking spiral and the critically damped one directly. The penalty turns divergence into convergence and leaves the equilibrium where it was.

### One-Centered versus Zero-Centered

WGAN-GP centers its penalty differently [@Gulrajani.Ahmed.Arjovsky.ea.2017]. To enforce the Lipschitz constraint of that section, it penalizes $\mathbb{E}\big[(\|\nabla_x D\| - 1)^2\big]$ at points between real and generated samples, because optimal $W_1$ critics have unit slope along transport rays. At $q = p$, however, the optimal critic is constant. A penalty that prefers unit slope keeps pushing a generator that already matches the data, and on the Dirac-GAN these dynamics do not converge [@Mescheder.Geiger.Nowozin.2018] (exercise 7).

| penalty | critic it prefers at $q = p$ | Dirac-GAN |
|:---|:---|:---|
| one-centered, $\mathbb{E}\big[(\|\nabla_x D\| - 1)^2\big]$ | unit slope | does not converge |
| zero-centered, $\mathbb{E}_m\big[\|\nabla_x D\|^2\big]$ | flat | converges for every $\gamma > 0$ |

### One Penalty or Two

The Dirac-GAN cannot tell $R_1$ from $R_2$, because a linear critic has the same slope everywhere. Near equilibrium the supports of $p$ and $q$ nearly coincide, so either penalty suffices, and the local convergence result of @Mescheder.Geiger.Nowozin.2018 needs only one. Far from equilibrium the two differ. @Roth.Lucchi.Nowozin.ea.2017 showed that penalizing $\mathbb{E}_p\big[\|\nabla_x D\|^2\big]$ changes the game, to leading order, as if $p$ were convolved with Gaussian noise of covariance proportional to $\gamma$, and $R_2$ smooths $q$ in the same way. Smoothed, two distributions with disjoint supports overlap, and the divergence responds to the distance between them. With $R_1$ alone the critic is unconstrained where only generated samples lie. R3GAN's ablation on StackedMNIST, stacked MNIST digits with 1,000 known modes [@Metz.Poole.Pfau.ea.2017], shows the consequence [@Huang.Gokaslan.Kuleshov.ea.2024]:

| loss | modes recovered (of 1,000) | reverse KL |
|:---|---:|---:|
| RpGAN with $R_1$ and $R_2$ | 1,000 | 0.078 |
| GAN with $R_1$ and $R_2$ | 693 | 0.927 |
| RpGAN with $R_1$ only | diverged | --- |
| GAN with $R_1$ only | diverged | --- |

With $R_1$ alone the critic's input gradients on generated samples grew and both losses diverged, for every $\gamma$ from 0.1 to 100 in the paired case. With both penalties both trained, and the paired loss recovered every mode: in this ablation the penalties decided convergence and the pairing decided coverage.

## The Penalized Value Near Equilibrium

The eigenvalues say how training approaches the equilibrium; what does the penalized game measure on the way? Without the penalty the value was a divergence of the density ratio, blind to how far apart separated supports are. With it, the value near equilibrium blends that divergence with a transport distance, and $\gamma$ decides which length scales fall on which side.

### The Local Value

Near $q = p$ the optimal critic is close to zero, so we expand the payoff to second order in the critic. With $\langle p - q, D \rangle = \mathbb{E}_p[D] - \mathbb{E}_q[D]$, and up to third-order terms,

$$
V_\ell(D) = 2\ell(0) + \ell'(0)\, \langle p - q,\, D \rangle - |\ell''(0)| \int m\, D^2 .
$$

The pairing objective has $\mathrm{Var}_m[D]$ in place of $\int m D^2$, which gives the same supremum once the critic's constant is chosen best. The quadratic term is not negligible: the best critic has size proportional to $p - q$, so all terms are of the same order at the optimum.

**Proposition.** *Let $\mathcal{L}_m D = -\nabla \cdot (m \nabla D)$. Near equilibrium, the penalized critic problem has the value*

$$
\sup_{D} \Big\{ \ell'(0)\, \langle p - q,\, D \rangle - |\ell''(0)| \int m\, D^2 - \gamma \int m\, \| \nabla_x D \|^2 \Big\}
\;=\; \frac{\ell'(0)^2}{4}\, \big\langle p - q,\, \big(|\ell''(0)|\, m + \gamma \mathcal{L}_m\big)^{-1} (p - q) \big\rangle,
$$

*attained by the critic that solves $\big(|\ell''(0)|\, m + \gamma \mathcal{L}_m\big) D = \tfrac{\ell'(0)}{2}(p - q)$.*

**Proof.** Write $\mathcal{A} = |\ell''(0)|\, m + \gamma \mathcal{L}_m$; integrating by parts, the two quadratic terms are $\langle D, \mathcal{A} D \rangle$, and $\mathcal{A}$ is symmetric and positive. Along a ray $\{t D_0\}$ the objective $t\,\ell'(0)\langle p - q, D_0\rangle - t^2 \langle D_0, \mathcal{A} D_0\rangle$ is a concave parabola in $t$ with maximum $\ell'(0)^2 \langle p - q, D_0\rangle^2 / (4\langle D_0, \mathcal{A} D_0 \rangle)$. By the Cauchy--Schwarz inequality in the inner product $\langle \cdot, \mathcal{A}\, \cdot \rangle$, the largest value over directions is $\tfrac{\ell'(0)^2}{4}\langle p - q, \mathcal{A}^{-1}(p - q)\rangle$, reached at $D_0 \propto \mathcal{A}^{-1}(p - q)$. $\blacksquare$

The value is a weighted squared size of $p - q$. As $\gamma \to 0$ it becomes $\tfrac{\ell'(0)^2}{4|\ell''(0)|}\int (p - q)^2/m$, which for the logistic loss is $\tfrac14 \int (p-q)^2/m$, the local form of $2\,\mathrm{JS}(p, q)$, the unpenalized value of that section. When $\gamma$ is large, the penalty dominates and the value approaches

$$
\frac{\ell'(0)^2}{4\gamma}\, \big\| p - q \big\|^2_{\dot H^{-1}(m)},
\qquad
\|v\|_{\dot H^{-1}(m)} = \sup\Big\{ \langle v, D\rangle : \int m \|\nabla_x D\|^2 \leq 1 \Big\},
$$

a squared dual Sobolev norm, taken over critics modulo constants.

### A Gaussian Shift

For $p = \mathcal{N}(0, 1)$ and $q = \mathcal{N}(\mu, 1)$ with a small shift $\mu$, to first order $p - q \approx -\mu\, x\, m(x)$. Since $\mathcal{L}_m x = x\, m(x)$ for the standard normal density $m$, a straight-line critic solves the equation of the proposition:

$$
D(x) = -\frac{\ell'(0)\,\mu}{2\,(|\ell''(0)| + \gamma)}\; x,
\qquad
\text{value} = \frac{\ell'(0)^2 \mu^2}{4\,(|\ell''(0)| + \gamma)} = \frac{\mu^2}{4 + 16\gamma}
\ \text{for the logistic loss}.
$$

This is the log ratio $-\mu x$ with its slope shrunk by $1 + 4\gamma$. At $\gamma = 0$ the value is $\mu^2/4 = 2\,\mathrm{JS}$ to leading order, and for large $\gamma$ it is $\mu^2/(16\gamma) = \tfrac{\ell'(0)^2}{4\gamma} W_2(p, q)^2$, since the Wasserstein-2 distance of a shift is $|\mu|$. To check both approximations, we solve the full penalized problem by Newton's method for a critic tabulated on a grid.

In [ ]:
def penalized_value(p, q, gamma, x, newton_steps=30):
    """Value of the logistic game minus gamma E_m[D'^2], on a grid."""
    dx, m = x[1] - x[0], (p + q) / 2
    diff = np.diff(np.eye(len(x)), axis=0) / dx       # D -> D'
    K = 2 * gamma * dx * diff.T @ (((m[1:] + m[:-1]) / 2)[:, None] * diff)
    D = np.zeros_like(x)
    for _ in range(newton_steps):                     # Newton ascent on D
        s = 1 / (1 + np.exp(-D))                      # sigma(D)
        grad = (p * (1 - s) - q * s) * dx - K @ D     # gradient of the value
        neg_hess = np.diag(2 * m * s * (1 - s) * dx) + K  # minus its Hessian
        D = D + np.linalg.solve(neg_hess, grad)
    value = -(p * np.logaddexp(0, -D) + q * np.logaddexp(0, D)).sum() * dx
    return value - D @ K @ D / 2 + 2 * np.log(2)       # minus 2 l(0)

x, mu = np.linspace(-8, 8, 401), 0.2
p, q = [np.exp(-(x - c) ** 2 / 2) / np.sqrt(2 * np.pi) for c in (0, mu)]
for gamma in (0, 0.01, 1, 10):
    large = mu ** 2 / (16 * gamma) if gamma else np.inf
    print(f'gamma {gamma:>5}: exact {penalized_value(p, q, gamma, x):.3e}, '
          f'local {mu ** 2 / (4 + 16 * gamma):.3e}, large-gamma {large:.3e}')

The local formula matches the exact value to within one percent at every $\gamma$. The large-$\gamma$ form holds only when the penalty dominates: at $\gamma = 0.01$ it gives $0.25$ against the exact $0.0096$, 26 times too large, and at $\gamma = 10$ it is within three percent.

### A Low-Pass Filter

A general $p - q$ has many length scales. Where $m$ is roughly constant, $\mathcal{L}_m$ multiplies a Fourier component $e^{i \omega x}$ by $m \omega^2$, so in the equation the component with wavenumber $\omega$ enters with weight

$$
\frac{1}{|\ell''(0)| + \gamma \omega^2}
\qquad \text{instead of} \qquad \frac{1}{|\ell''(0)|}.
$$

Components whose length scale $1/\omega$ exceeds $\sqrt{\gamma / |\ell''(0)|}$, which is $2\sqrt\gamma$ for the logistic loss, keep nearly their unpenalized weight; shorter ones are damped by about the factor $|\ell''(0)|/(\gamma \omega^2)$. The penalty is a low-pass filter on $p - q$ with cutoff length $2\sqrt\gamma$. Above the cutoff the game measures $2\,\mathrm{JS}$; below it, the dual Sobolev norm of the equation, which equals $W_2(p, q)$ to leading order as $q \to p$ by the Benamou--Brenier form of optimal transport (that section). Below the cutoff the generator therefore keeps a signal when nearby supports move apart. Because the critic's input gradient has units of one over length, $\gamma$ is a squared length, set by the scale of the data: R3GAN uses values from 0.05 on CIFAR-10 to 150 on FFHQ at $256 \times 256$ pixels.

## The R3GAN Recipe

R3GAN [@Huang.Gokaslan.Kuleshov.ea.2024] puts the pieces together. Its critic minimizes the pairing loss of that section plus both zero-centered penalties, and its generator minimizes the non-saturating form of that loss. The penalties make the equilibrium locally attracting; the pairing contributed the coverage measured in the StackedMNIST ablation above. With this loss R3GAN trains a plain convolutional network without most of the stabilizers that earlier GANs needed (that section).

### The Loss in Code

The implementation keeps the losses and the penalties apart. For aligned pairs of a real sample $x$ and a generated sample $x'$, the critic minimizes $\mathrm{softplus}(D(x') - D(x)) = -\log \sigma(D(x) - D(x'))$ and the generator the reversed $\mathrm{softplus}(D(x) - D(x'))$. The functions `gan_loss_D` and `gan_loss_G` give the logistic critic loss and the non-saturating generator loss of that section with the same arguments, so a training loop can take either pair.

In [ ]:

def rpgan_loss_D(critic, real, fake):
    """Relativistic pairing critic loss -E[log sigma(D(x) - D(x'))]."""
    return jax.nn.softplus(critic(fake) - critic(real)).mean()


def rpgan_loss_G(critic, real, fake):
    """Non-saturating pairing loss for the generator."""
    return jax.nn.softplus(critic(real) - critic(fake)).mean()


def gan_loss_D(critic, real, fake):
    """Logistic critic loss -E[log sigma(D(x))] - E[log sigma(-D(x'))]: the
    batch means, where update_D reports the halved sum."""
    return (jax.nn.softplus(-critic(real)).mean()
            + jax.nn.softplus(critic(fake)).mean())


def gan_loss_G(critic, real, fake):
    """Non-saturating generator loss -E[log sigma(D(x'))]; real is unused."""
    return jax.nn.softplus(-critic(fake)).mean()

The penalty function returns the per-sample squared input gradients on both batches, before the factor $\gamma/2$. It differentiates the *sum* of the critic's scores with respect to the input batch: each score depends only on its own row, so one backward pass gives every sample's input gradient (batch normalization, which mixes rows, would break this). The gradient stays differentiable, so that training can differentiate the penalty with respect to the critic's parameters; the samples are constants. The critic may be any function from a batch of inputs to a batch of scores, including a closure that fixes a condition (that section).

`jax.grad` of the summed scores gives the input gradients; differentiating the penalty with respect to the critic nests a second gradient around it, and `jax.lax.stop_gradient` makes the samples constants.

In [ ]:

def r1_r2_penalty(critic, real, fake):
    """Per-sample squared critic input gradients on real (R1) and fake (R2),
    before the gamma/2 scale."""
    def sq_grad_norm(x):
        # Rows are independent, so the gradient of the sum is per-sample
        grad = jax.grad(lambda x: critic(x).sum())(jax.lax.stop_gradient(x))
        return (grad.reshape(x.shape[0], -1) ** 2).sum(axis=1)
    return sq_grad_norm(real), sq_grad_norm(fake)

### The Training Steps

A training loop alternates one step of each player. `critic_step` takes one optimizer step on the critic loss plus $\gamma/2$ times both penalties, with the generated batch held fixed, and returns the loss and the batch means of $R_1$ and $R_2$, so that a loop can log the critic's steepness. `generator_step` then takes one step on the generator loss against the updated critic. Given labels `y`, both score with the conditional critic $D(\cdot, y)$ (that section).

In [ ]:

@nnx.jit(static_argnums=2)
def critic_step(net_D, opt_D, loss_D, real, fake, gamma, y=None):
    """One critic step on loss_D + gamma/2 (R1 + R2); return the loss and
    the batch means of R1 and R2."""
    def objective(net_D):
        critic = net_D if y is None else lambda x: net_D(x, y)
        r1, r2 = r1_r2_penalty(critic, real, fake)
        loss = loss_D(critic, real, fake)
        return (loss + gamma / 2 * (r1 + r2).mean(),
                (loss, r1.mean(), r2.mean()))
    (_, logs), grads = nnx.value_and_grad(objective, has_aux=True)(net_D)
    opt_D.update(net_D, grads)
    return logs


@nnx.jit(static_argnums=2)
def generator_step(net_G, opt_G, loss_G, net_D, real, z, y=None):
    """One generator step on loss_G against the critic; return the loss."""
    def objective(net_G):
        critic = net_D if y is None else lambda x: net_D(x, y)
        return loss_G(critic, real, net_G(z) if y is None else net_G(z, y))
    loss, grads = nnx.value_and_grad(objective)(net_G)
    opt_G.update(net_G, grads)
    return loss

The loss functions are static arguments of the compiled steps, so each pair of losses compiles once, while $\gamma$ is traced and can change without recompiling.

### Mode Coverage on 25 Gaussians

We test the recipe on a standard problem with many modes: 25 Gaussians whose centers form a $5 \times 5$ grid with spacing 2, each with standard deviation $0.05$. A generated sample is *on a mode* if it lies within three widths, $0.15$, of the nearest center, where a 2-D Gaussian puts $1 - e^{-9/2} \approx 98.9\%$ of its mass. For 10,000 samples we report how many modes are covered, counting a mode only if it holds at least a fifth of the on-mode samples it would receive if they were spread evenly, then the fraction of samples on a mode, their mean distance to the nearest center (about $0.06$ for a perfect fit), and $\mathbb{E}_q\|\nabla_x D\|^2$ for the final critic. Both networks are MLPs with two hidden layers of 256 units, built by `d2l.make_mlp` of that section; the generator maps a 64-dimensional latent vector to the plane, and the critic uses leaky ReLU and no normalization.

In [ ]:
def grid_centers(side=5, spacing=2.0):
    coords = (jnp.arange(side) - (side - 1) / 2) * spacing
    gx, gy = jnp.meshgrid(coords, coords, indexing='ij')
    return jnp.stack([gx.reshape(-1), gy.reshape(-1)], axis=1)

def sample_modes(key, centers, n, width=0.05):
    k1, k2 = jax.random.split(key)
    idx = jax.random.randint(k1, (n,), 0, centers.shape[0])
    return centers[idx] + width * jax.random.normal(k2, (n, 2))

The training function takes the two losses and $\gamma$ as arguments and alternates `critic_step` and `generator_step`, each on a fresh batch. Both networks use Adam with $\beta_1 = 0$ and $\beta_2 = 0.99$, as R3GAN does, and a learning rate that decays from $2 \cdot 10^{-3}$ to $2 \cdot 10^{-4}$ along a cosine over 10,000 steps with batches of 1,024. It returns the generator and $\mathbb{E}_q\|\nabla_x D\|^2$ from the last critic step, which at $\gamma = 0$ is the only use of the penalties.

In [ ]:
def train_grid(centers, loss_D, loss_G, gamma, steps=10000, lr=2e-3,
               batch=1024, latent=64, seed=0):
    rngs = nnx.Rngs(seed)
    net_G = d2l.make_mlp([latent, 256, 256, 2], nnx.relu, rngs=rngs)
    net_D = d2l.make_mlp([2, 256, 256, 1], lambda x: nnx.leaky_relu(x, 0.2),
                         rngs=rngs)
    adam = optax.adam(optax.cosine_decay_schedule(lr, steps, alpha=0.1),
                      b1=0.0, b2=0.99)
    opt_G, opt_D = (nnx.Optimizer(net, adam, wrt=nnx.Param)
                    for net in (net_G, net_D))

    @nnx.jit
    def step(net_G, net_D, opt_G, opt_D, key):
        k1, k2 = jax.random.split(key)
        z = jax.random.normal(k1, (2, batch, latent))    # critic, generator
        real = sample_modes(k2, centers, 2 * batch).reshape(2, batch, 2)
        fake = net_G(z[0])
        _, _, grad_q = critic_step(net_D, opt_D, loss_D, real[0], fake, gamma)
        generator_step(net_G, opt_G, loss_G, net_D, real[1], z[1])
        return grad_q                      # E_q ||grad_x D||^2

    step = nnx.cached_partial(step, net_G, net_D, opt_G, opt_D)
    for key in jax.random.split(rngs(), steps):
        grad_q = step(key)
    return net_G, float(grad_q)

One compiled function draws the batches and calls both saved steps. `nnx.cached_partial` binds the networks and optimizers to it once, so the loop does not split and merge the model state at every step.

We train three runs from the same initialization: the classical GAN without a penalty, and the R3GAN loss at $\gamma = 1$ and at $\gamma = 10$. The low-pass filter predicts how the last two differ. The grid repeats every 2 units, a component with wavenumber $\omega = \pi$, which the penalty damps about 40-fold at $\gamma = 1$ and about 400-fold at $\gamma = 10$. With a ten times weaker signal about where the modes sit, the $\gamma = 10$ generator should resolve the grid less sharply within the same budget.

In [ ]:
centers = grid_centers()
arms = {'GAN': (gan_loss_D, gan_loss_G, 0.0),
        'R3GAN, gamma=1': (rpgan_loss_D, rpgan_loss_G, 1.0),
        'R3GAN, gamma=10': (rpgan_loss_D, rpgan_loss_G, 10.0)}
latent, samples, grad_q = 64, {}, {}
for name, (loss_D, loss_G, gamma) in arms.items():
    net_G, grad_q[name] = train_grid(centers, loss_D, loss_G, gamma,
                                     latent=latent)
    z = jax.random.normal(jax.random.key(1), (10000, latent))
    samples[name] = np.asarray(net_G(z))
grid = np.asarray(centers)

Each panel shows 3,000 generated samples in orange and the 25 centers in blue.

In [ ]:
def mode_stats(x, centers, width=0.05):
    """Modes covered (each holds a fifth of an even share of the on-mode
    samples), fraction within three widths, mean distance."""
    dist = np.linalg.norm(x[:, None] - centers[None], axis=-1)
    near, idx = dist.min(axis=1), dist.argmin(axis=1)
    on = near <= 3 * width
    counts = np.bincount(idx[on], minlength=len(centers))
    covered = (counts >= on.sum() / len(centers) / 5).sum()
    return covered, on.mean(), near.mean()

fig, axes = d2l.plt.subplots(1, 3, figsize=(9.5, 3.4))
for ax, (name, x) in zip(axes, samples.items()):
    modes, on, dist = mode_stats(x, grid)
    print(f'{name}: {modes}/25 modes, on-mode {on:.2f}, '
          f'distance {dist:.2f}, E_q|grad D|^2 {grad_q[name]:.1e}')
    # Centers drawn last, on top of the samples
    d2l.plot([x[:3000, 0], grid[:, 0]], [x[:3000, 1], grid[:, 1]],
             fmts=('C1.', 'C0x'), xlim=(-5, 5), ylim=(-5, 5), axes=ax)
    ax.set_title(name)
    ax.set_aspect('equal')
fig.tight_layout()

The R3GAN loss at $\gamma = 1$ fits the grid. Over sixteen runs with different seeds it covered all 25 modes in every run but one, which covered 24, and put most of its samples within three widths of a center. The classical GAN also puts most of its samples on a mode but spreads them less evenly. Over about 150 runs it covered all 25 modes in only about two runs out of five, and in more than a quarter of them it left a whole row or column of the grid nearly empty. The two losses differ more reliably in the critic. With the penalty, $\mathbb{E}_q\|\nabla_x D\|^2$ ends about three to four orders of magnitude below its value for the unpenalized critic, which stays steep at the generated samples even when the fit is good, like the undamped slope $\psi$ of the Dirac-GAN. At $\gamma = 10$ the grid is blurred, as predicted: only an eighth to three fifths of the samples lie on a mode, and their mean distance to the nearest center is 1.5 to 6 times that at $\gamma = 1$. The blur also thins out some modes, mostly near the center of the grid, so the strict count finds between 21 and 25 covered. The equilibrium is still $q = p$; the larger $\gamma$ slows the approach to it (exercise 4).

## The Games of This Chapter

### Three Choices

Four sections have now derived what each adversarial game measures. Three choices describe all of them (the table): what the critic estimates, what it does with the estimate, and how the critic is constrained. The last column records whether the generator keeps a gradient when the supports of $p$ and $q$ are disjoint.

<!-- tbl-caption:The games of this chapter, described by three choices: what the critic estimates, what it does with the estimate, and how the critic is constrained. The last column records whether the generator keeps a gradient on disjoint supports. -->

| objective | the critic estimates | what it does with the estimate | how the critic is constrained | on disjoint supports |
|:--|:--|:--|:--|:--|
| log loss, $f$-GAN (that section, that section) | $\log(p/q)$, or $f'(p/q)$ for an $f$-GAN | classifies one sample | not at all | saturates |
| MMD (that section) | a kernel-smoothed $p - q$, the witness | witnesses a difference of means | a fixed kernel ball | smooth |
| Wasserstein-1 (that section) | a Kantorovich potential | witnesses a difference of means | a Lipschitz ball | smooth |
| pairing, RpGAN (that section) | $\log(p/q)$ up to a constant | ranks a real--generated pair | not at all | saturates |
| RpGAN + $R_1$ + $R_2$ (that section) | $\log(p/q)$ above the length scale $2\sqrt\gamma$, smoothed below | ranks a real--generated pair | a symmetric Sobolev penalty | smooth, near equilibrium |

### Reading the Table

The classifying and the ranking critics estimate the same log ratio; the loss and the pairing change how its errors are weighted, not what is estimated. The MMD and $W_1$ critics estimate a smoothed difference and a potential, and the penalty moves the pairing critic toward them below the length scale $2\sqrt\gamma$. The constraint decides whether a gradient survives on disjoint supports, and the zero-centered penalty also damps the rotation of the Dirac-GAN. Ranking guarantees coverage only for a fixed coupling; for minibatch training the evidence is the StackedMNIST ablation, where the pairing loss recovered all 1,000 modes and the classical loss 693.

Whether this recipe trains an image generator, and how to score the images it produces, is the question of that section.

## Summary

Gradient training of an adversarial game need not reach its equilibrium. On the Dirac-GAN the gradient field is orthogonal to the parameters, so the flow circles the solution and every simultaneous step moves away from it, whatever the loss, its weighting or its pairing. The zero-centered penalties $R_1$ and $R_2$ on the critic's input gradient at data and generated samples damp the rotation for every $\gamma > 0$. Near equilibrium the penalized game measures $p - q$ through a low-pass filter with cutoff length $2\sqrt\gamma$: $2\,\mathrm{JS}$ at larger scales, a linearized Wasserstein-2 distance at smaller ones. R3GAN combines the paired loss with both penalties; on 25 Gaussians it fits the grid at $\gamma = 1$ with a critic that stays flat on generated samples, while $\gamma = 10$ blurs the grid within the same budget.

## Exercises

1. [code] **Discrete-time stability.** A simultaneous step of size $h$
   maps $(\theta, \psi)$ to $(\theta, \psi) + h\, \mathbf{u}(\theta, \psi)$,
   with $\mathbf{u}$ from the equation plus $-\gamma\psi$ in
   its second component for a penalized critic.
    1. Show that at $\gamma = 0$ the Jacobian of the map at the
       equilibrium has eigenvalues $1 \pm i\, h\, \ell'(0)$, so no step
       size makes the unpenalized game locally convergent.
    1. For $\gamma > 0$ the eigenvalues are $1 + h\nu$, with $\nu$
       from the equation. Show that in the underdamped regime
       $\gamma < 2\ell'(0)$ the iteration converges locally if and only if
       $h < \gamma / \ell'(0)^2$.
    1. Check the threshold of (b) at $\gamma = 0.3$: run
       `trajectory((0.1, 0.1), h, 3000, gamma=0.3)` for $h = 1.1$ and
       $h = 1.3$ and plot the distance from the equilibrium. What does the
       iteration do above the threshold?
1. **Penalized best response.** Let the critic of the Dirac-GAN subtract
   both penalties of the equation, which for the linear critic sum
   to $\gamma\psi^2$, take $\ell = \log\sigma$, and write
   $V_\gamma(\theta) = \sup_\psi \big\{ V(\theta, \psi) - \gamma\psi^2 \big\}$.
    1. Show that the supremum is attained for every $\theta$, and that for
       small $\theta$ the maximizer is
       $\psi^\star \approx -\ell'(0)\,\theta / (2\gamma)$ and
       $V_\gamma(\theta) = 2\ell(0) + \tfrac{\ell'(0)^2}{4\gamma}\, \theta^2 + O(\theta^4)$.
    1. Compare (a) with the equation, using
       $W_2(\delta_0, \delta_\theta) = |\theta|$. Why does the curvature
       term of the equation not appear at order $\theta^2$?
       What gradient does the generator receive near $\theta = 0$, and how
       does it depend on $\gamma$?
    1. Now let the critic be any differentiable function, penalized by
       $\gamma \int m\, \|\nabla_x D\|^2$. Show that the penalized
       best-response value is $2\sup_t \ell(t)$ for every $\theta \neq 0$.
       Which assumption behind the equation fails here?
1. **Penalizing one side only.** Let $p = \delta_0$, put the generated
   point at $x' = 1$, take $\ell = \log\sigma$, and consider the critics
   $D_c(x) = -c\, x^2$ with $c \geq 0$, flat at the data and steep at the
   generated point.
    1. Evaluate $R_1$ and $R_2$ for $D_c$. Show that the critic objective
       $V_\ell(D_c) - R_1$ increases with $c$ and has no maximizer.
    1. Show that the gradient of the non-saturating generator loss
       $-\log\sigma(D(x'))$ with respect to $x'$ at $x' = 1$ grows without
       bound in $c$. What does the saturating loss $\log\sigma(-D(x'))$
       give instead? Which observation of
       @Huang.Gokaslan.Kuleshov.ea.2024 does this reproduce?
    1. Subtract $R_1 + R_2$ instead. Show that the maximizing $c^\star$ is
       finite with $c^\star < 1/(8\gamma)$, and bound the generator's
       gradient at $c^\star$.
1. [code] **The penalty weight as a length.** In
   that section, $\gamma = 10$ blurs the grid that
   $\gamma = 1$ fits.
    1. For data scaled by $c > 0$ and the critic $D_c(x) = D(x / c)$, show
       that the penalized game at weight $c^2\gamma$ has the same value as
       the original game at weight $\gamma$.
    1. With the low-pass weight and $|\ell''(0)| = \tfrac14$, compute by
       what factor the penalty damps a component whose wavelength is the
       mode width $0.05$, at $\gamma = 1$ and at $\gamma = 10$. Compare
       the mean distance printed for $\gamma = 1$ with the $0.06$ of a
       perfect fit. Does the damping predict how sharply each mode is
       resolved?
    1. Train the $\gamma = 10$ run for two and four times as many steps,
       with the cosine schedule stretched to match. Does the on-mode
       fraction approach that of $\gamma = 1$?
1. [code] **Alternating updates.** ● `train_grid` updates the critic first
   and then the generator against the updated critic. On the Dirac-GAN
   this reads

   $$\psi_{k+1} = \psi_k + h\, u_\psi(\theta_k, \psi_k), \qquad \theta_{k+1} = \theta_k + h\, u_\theta(\theta_k, \psi_{k+1}),$$

   with $(u_\theta, u_\psi)$ the field of the equation.
    1. Linearize at the equilibrium for $\gamma = 0$ and show that the
       Jacobian of the combined update has determinant one, with
       eigenvalues on the unit circle if and only if $h\, \ell'(0) \leq 2$.
       What does this imply about linear convergence and divergence?
    1. Write an alternating `trajectory` and run it at $\gamma = 0$ from
       $(0.1, 0.1)$ for $10^5$ steps with $h \in \{0.2, 1, 3, 5\}$. Record
       the smallest and largest distance from the equilibrium over the
       last $10^4$ steps, and explain the wide spread at $h = 3$. Repeat at
       $h = 0.2$ with $\gamma = 0.3$.
1. [code] **The local value.** ●
    1. Perturb the maximizing critic of the equation to
       $D + s\,\xi$ with a test function $\xi$ and show that the
       first-order condition in $s$ is
       $\big(|\ell''(0)|\, m + \gamma \mathcal{L}_m\big) D = \tfrac{\ell'(0)}{2}(p - q)$,
       assuming that boundary terms vanish.
    1. Let $p = \mathcal{N}(0, 1)$ and $q = p\,(1 + 0.1 \sin \omega x)$. Use
       `penalized_value` to compute the ratio of the values at $\gamma$
       and at $\gamma = 0$ for $\omega \in \{0.5, 2, 8\}$ and
       $\gamma \in \{0.01, 0.1, 1\}$, and compare it with the low-pass
       weight $1/(1 + 4\gamma \omega^2)$. For which $\omega$ does the
       prediction fail, and which assumption of the Fourier argument breaks
       there?
1. [code] **One-centered penalty dynamics.** ● Take the linear score
   $\ell(t) = t$ of the Wasserstein critic, so $\ell'(0) = 1$. On the
   Dirac-GAN the one-centered penalty becomes
   $\tfrac{\gamma}{2}(|\psi| - 1)^2$, which the critic subtracts.
    1. Show that the simultaneous gradient field vanishes at no point
       with $\psi \neq 0$, and find where its one-sided limits as
       $\psi \to 0^\pm$ vanish. Where is the generator there?
    1. On $\psi > 0$ the field is affine. Compare its Jacobian with
       $J_\gamma$ of the equation; for which $\gamma$ are its
       eigenvalues real?
    1. Modify `dirac_field` and run `trajectory` from $(1, 1)$ with
       $h = 0.01$ for 40,000 steps at $\gamma = 1$ and $\gamma = 3$. Where
       does each trajectory end? Explain both outcomes with (a) and (b).
1. [code] **Momentum in a rotating game.** ● R3GAN trains with Adam at
   $\beta_1 = 0$. To isolate the first moment, average the field of the
   penalized Dirac-GAN with the logistic loss,
   $\bar{\mathbf{u}}_k = \beta\, \bar{\mathbf{u}}_{k-1} + (1 - \beta)\, \mathbf{u}(\mathbf{x}_k)$
   and $\mathbf{x}_{k+1} = \mathbf{x}_k + h\, \bar{\mathbf{u}}_k$, with
   $\mathbf{x}_k = (\theta_k, \psi_k)$ and $\bar{\mathbf{u}}_{-1} = \mathbf{0}$.
   Linearize at the equilibrium, write one step as a linear map of
   $(\mathbf{x}_k, \bar{\mathbf{u}}_{k-1}) \in \mathbb{R}^4$, and compute
   numerically the largest stable step size for $\beta \in \{0, 0.5, 0.9\}$
   at $\gamma = 0.3$ and $\gamma = 1$. Check $\beta = 0$ against problem
   1b. In which
   damping regime does momentum shrink the range of stable step sizes?
1. [code] **Non-saturating Dirac dynamics.** ●● The non-saturating
   generator replaces $\ell'(-\psi\theta)$ by $\ell'(\psi\theta)$ in the
   first component of the equation. Take
   $\ell = \log\sigma$, $\gamma = 0$ and $r^2 = \theta^2 + \psi^2$.
    1. Show that
       $\tfrac{d}{dt} r^2 = 2\psi\theta\,\big(\ell'(\psi\theta) - \ell'(-\psi\theta)\big) \leq 0$
       for a concave $\ell$. Why can the approach not be linear?
    1. Average $\tfrac{d}{dt} r^2$ over one turn, at angular rate
       $\ell'(0)$, to leading order in $r$, and show that
       $r(t)^2 \approx 8/t$ for large $t$.
    1. One simultaneous step changes $r^2$ by
       $2h\, \langle \mathbf{x}, \mathbf{u} \rangle + h^2 \|\mathbf{u}\|^2$.
       Average this over one turn and show that it vanishes at
       $(r^\star)^2 = 2h$. Is this cycle stable?
    1. Redefine `dirac_field` with the non-saturating weight, run
       `trajectory` from $(1, 1)$ and from $(0.05, 0.05)$ for $4000/h$
       steps with $h \in \{0.01, 0.05, 0.2\}$, and compare the late
       distance with $r^\star$. Check (b) with `flow=True`.

[Discussions](https://d2l.discourse.group/t/36811)